<p style="text-align: center">
<img src="../../assets/images/dtlogo.png" alt="Duckietown" width="50%">
</p>

# Guida Completa ed Intuitiva al Filtro di Kalman Esteso (EKF Localization)
### Localizzazione basata su Mappa e Riferimenti Visivi (AprilTags) nel Duckiebot

Benvenuto nel quarto e fondamentale notebook della serie sulla localizzazione robotica!
Nel primo notebook abbiamo imparato come funziona il **Filtro di Kalman lineare (KF)**. Tuttavia, in quel modello avevamo fatto un'assunzione ideale:
$$\mathbf{x}_t = A \mathbf{x}_{t-1} + B \mathbf{u}_t, \quad \mathbf{z}_t = H \mathbf{x}_t$$
Nel mondo reale della robotica mobile, **i sistemi fisici NON sono quasi mai lineari**!

---

### Perché i robot sono intrinsecamente "non lineari"?

1. **Il Movimento include l'Orientamento (Angolo $\theta$):**
   * Un Duckiebot non si muove solo lungo $x$ e $y$: ha un angolo di orientamento $\theta$ (la sua direzione sul pavimento).
   * Se il robot avanza di una distanza $dX$, le sue coordinate cartesiane cambiano secondo funzioni trigonometriche:
     $$\Delta x = dX \cos(\theta), \quad \Delta y = dX \sin(\theta)$$
   * Il seno e il coseno sono funzioni **curve e non lineari**. Non puoi scriverle come una semplice moltiplicazione $A \cdot \mathbf{x}$!

2. **I Sensori misurano in Coordinate Polari:**
   * La telecamera del Duckiebot rileva cartelli stradali con codici a barre 2D chiamati **AprilTags** (landmark o punti di riferimento noti sulla mappa).
   * La misura visiva restituisce due grandezze:
     * **Range ($r$):** la distanza in linea d'aria dal robot al cartello $\implies r = \sqrt{(l_x - x)^2 + (l_y - y)^2}$
     * **Bearing ($\phi$):** l'angolo visivo del cartello rispetto al muso del robot $\implies \phi = \text{atan2}(l_y - y, l_x - x) - \theta$
   * La radice quadrata e l'arcotangente sono funzioni **estremamente non lineari**!

---

### Il "Colpo di Genio" dell'EKF: La Linearizzazione con le Matrici Jacobiane
Se provi a far passare una campana gaussiana attraverso una funzione curva come $\sin(\theta)$ o $\sqrt{x^2+y^2}$, la campana si deforma e cessa di essere gaussiana: il Filtro di Kalman classico fallirebbe.

L'**Extended Kalman Filter (EKF)** risolve il problema usando la **Serie di Taylor al primo ordine**:
> Invece di considerare l'intera curva complessa, ad ogni istante di tempo calcoliamo la **retta tangente** (la derivata parziale) attorno alla stima attuale!

Queste tabelle di derivate parziali si chiamano **Matrici Jacobiane ($F, W, H$)** e sostituiscono le vecchie matrici costanti $A, B, H$ del filtro lineare.

In questo notebook smonteremo ogni singola equazione, ricaveremo le derivate passo dopo passo, implementeremo il codice per `ekf.py` e simuleremo un Duckiebot che naviga e si localizza con gli AprilTags!

In [ ]:
%matplotlib inline
import time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Fissiamo il seed casuale per garantire riproducibilita
np.random.seed(42)
print("Librerie importate con successo!")

---
# Passo 1: La Mappa e la Definizione dello Stato del Robot

### 1. La Mappa dei Landmark (AprilTags)
Nel sistema di navigazione di Duckietown, la pista è dotata di cartelli stradali provvisti di AprilTag.
Le coordinate esatte $(l_x, l_y)$ di ciascun cartello nel mondo sono scritte nel file di configurazione `packages/ekf_localization/map.yaml`:
```yaml
map:
  "1":
    position: [0.6786, 1.76085]
  "20":
    position: [-0.02925, 1.8252]
  ...
```
* L'origine $(0,0)$ del mondo nella simulazione (Duckiematrix) vede il Duckiebot partire orientato verso l'asse $+y$, con l'asse $+x$ che punta a destra.

### 2. Il Vettore di Posa del Robot: `self.q`
Lo stato del Duckiebot è memorizzato in un vettore tridimensionale:
$$\mathbf{q} = \begin{bmatrix} x \\ y \\ \theta \end{bmatrix}$$
* $x, y$: posizione sul pavimento (in metri).
* $\theta$: angolo di orientamento (in radianti). Usiamo sempre la funzione `wrap_angle` per garantire che $\theta \in [-\pi, \pi]$.

### 3. La Matrice di Covarianza dello Stato: `self.P`
L'incertezza sulla nostra posa è una matrice $3 \times 3$:
$$P = \begin{bmatrix} P_{xx} & P_{xy} & P_{x\theta} \\ P_{xy} & P_{yy} & P_{y\theta} \\ P_{x\theta} & P_{y\theta} & P_{\theta\theta} \end{bmatrix}$$
* Sulla diagonale troviamo le varianze delle coordinate $x, y$ e dell'orientamento $\theta$.
* Fuori diagonale troviamo le correlazioni tra posizione e orientamento.

In [ ]:
def wrap_angle(angle):
    """
    Normalizza l'angolo nell'intervallo [-pi, +pi].
    Fondamentale in robotica per evitare ambiguita (es. +181 gradi deve diventare -179 gradi).
    """
    return (angle + np.pi) % (2.0 * np.pi) - np.pi

# Test rapido di wrap_angle
print("wrap_angle(3.5 rad) =", wrap_angle(3.5), "rad (circa", np.degrees(wrap_angle(3.5)), "gradi)")

---
# Passo 2: La Fase di Predizione (*Predict Step*) — Cinematica e Jacobiane

La funzione di predizione:
```python
def predict(self, dX, dT):
```
riceve in ingresso due quantità calcolate dai "tick" degli encoder delle ruote:
* **`dX`**: l'avanzamento lineare lungo l'asse del robot (in metri) dall'ultimo passo;
* **`dT`**: la rotazione angolare compiuta (in radianti) dall'ultimo passo.

---

### Step 1: Propagazione dello Stato col Modello Cinematico Non Lineare $f$
Dall'istante precedente $\mathbf{q}_{t-1} = [x, y, \theta]^T$, la nuova stima è data da:
$$\begin{cases} x_t = x_{t-1} + dX \cos(\theta_{t-1}) \\ y_t = y_{t-1} + dX \sin(\theta_{t-1}) \\ \theta_t = \text{wrap\_angle}(\theta_{t-1} + dT) \end{cases}$$

---

### Step 2: Calcolo delle Matrici Jacobiane del Processo ($F$ e $W$)
Poiché $f$ non è lineare, calcoliamo le derivate parziali rispetto allo stato $\mathbf{q}$ e rispetto agli ingressi rumorosi $\mathbf{u} = [dX, dT]^T$:

1. **La Jacobiana dello Stato ($F = \frac{\partial f}{\partial \mathbf{q}}$, matrice $3 \times 3$):**
   $$F = \begin{bmatrix} 
   \frac{\partial x_t}{\partial x} & \frac{\partial x_t}{\partial y} & \frac{\partial x_t}{\partial \theta} \\[6pt]
   \frac{\partial y_t}{\partial x} & \frac{\partial y_t}{\partial y} & \frac{\partial y_t}{\partial \theta} \\[6pt]
   \frac{\partial \theta_t}{\partial x} & \frac{\partial \theta_t}{\partial y} & \frac{\partial \theta_t}{\partial \theta}
   \end{bmatrix} = 
   \begin{bmatrix}
   1 & 0 & -dX \sin(\theta) \\
   0 & 1 & dX \cos(\theta) \\
   0 & 0 & 1
   \end{bmatrix}$$
   *(Guarda l'elemento $F[0, 2] = -dX \sin(\theta)$: esprime matematicamente come un errore sull'angolo si amplifica sulla coordinata $x$ in proporzione a quanto velocemente avanzi!)*

2. **La Jacobiana del Controllo/Rumore ($W = \frac{\partial f}{\partial \mathbf{w}}$, matrice $3 \times 2$):**
   $$W = \begin{bmatrix}
   \frac{\partial x_t}{\partial dX} & \frac{\partial x_t}{\partial dT} \\[6pt]
   \frac{\partial y_t}{\partial dX} & \frac{\partial y_t}{\partial dT} \\[6pt]
   \frac{\partial \theta_t}{\partial dX} & \frac{\partial \theta_t}{\partial dT}
   \end{bmatrix} =
   \begin{bmatrix}
   \cos(\theta) & 0 \\
   \sin(\theta) & 0 \\
   0 & 1
   \end{bmatrix}$$

---

### Step 3: Aggiornamento della Covarianza Predetta
Proiettiamo la covarianza precedente $P$ usando la forma linearizzata e sommiamo la covarianza del rumore di processo $Q = \text{diag}(Q_{xx}, Q_{\theta\theta})$:
$$P_t = F P_{t-1} F^T + W Q W^T$$

---
# Passo 3: La Fase di Aggiornamento (*Update Step*) — Misure AprilTag

Quando la telecamera del Duckiebot vede un AprilTag, il driver rileva:
* $z = [r, \phi]^T$ (la misura reale di distanza e angolo nel frame del robot).
* $\text{tag\_xy} = [l_x, l_y]^T$ (la posizione nota del cartello sulla mappa).

---

### Step 1: Calcolo della Misura Predetta $h(\mathbf{q})$
Dove *dovrebbe* trovarsi il cartello secondo la nostra attuale stima della posa $[x, y, \theta]^T$?
1. Calcoliamo la distanza vettoriale: $\Delta x = l_x - x$, $\Delta y = l_y - y$.
2. Distanza predetta:
   $$r_{\text{pred}} = \sqrt{\Delta x^2 + \Delta y^2}$$
3. Angolo visivo predetto (nel riferimento solidale al veicolo):
   $$\phi_{\text{pred}} = \text{wrap\_angle}\left(\text{atan2}(\Delta y, \Delta x) - \theta\right)$$
   $$z_{\text{pred}} = \begin{bmatrix} r_{\text{pred}} \\ \phi_{\text{pred}} \end{bmatrix}$$

---

### Step 2: Calcolo dell'Innovazione (Residuo $y$)
$$y = z - z_{\text{pred}} = \begin{bmatrix} r - r_{\text{pred}} \\ \text{wrap\_angle}(\phi - \phi_{\text{pred}}) \end{bmatrix}$$

---

### Step 3: Calcolo della Jacobiana di Misura ($H = \frac{\partial h}{\partial \mathbf{q}}$, matrice $2 \times 3$)
Deriviamo $r$ e $\phi$ rispetto a $x, y, \theta$:
$$H = \begin{bmatrix}
\frac{\partial r}{\partial x} & \frac{\partial r}{\partial y} & \frac{\partial r}{\partial \theta} \\[6pt]
\frac{\partial \phi}{\partial x} & \frac{\partial \phi}{\partial y} & \frac{\partial \phi}{\partial \theta}
\end{bmatrix} = 
\begin{bmatrix}
-\frac{\Delta x}{r_{\text{pred}}} & -\frac{\Delta y}{r_{\text{pred}}} & 0 \\[8pt]
\frac{\Delta y}{r_{\text{pred}}^2} & -\frac{\Delta x}{r_{\text{pred}}^2} & -1
\end{bmatrix}$$

---

### Step 4 & 5: Guadagno di Kalman, Correzione di Posa e Covarianza
1. **Covarianza dell'innovazione:**
   $$S = H P H^T + R \quad (2 \times 2)$$
2. **Guadagno di Kalman:**
   $$K = P H^T S^{-1} \quad (3 \times 2)$$
3. **Aggiornamento della posa stimata:**
   $$\mathbf{q} = \mathbf{q} + K y, \quad \mathbf{q}[2] = \text{wrap\_angle}(\mathbf{q}[2])$$
4. **Aggiornamento della covarianza:**
   $$P = (I - K H) P$$

---
# Implementazione Completa della Classe `EKF`

Ecco il codice Python completo e collaudato che implementa l'intera architettura EKF descritta per `packages/solution/ekf.py`:

In [ ]:
class EKF:
    def __init__(self, q_0: np.ndarray, P_0: np.ndarray, Q: np.ndarray, R: np.ndarray):
        """
        Inizializza il Filtro di Kalman Esteso.
        q_0: Vettore di posa iniziale [x, y, theta]
        P_0: Matrice di covarianza iniziale (3x3)
        Q: Covarianza rumore di processo comandi (2x2: diag(Q_xx, Q_tt))
        R: Covarianza rumore sensore AprilTag (2x2: diag(R_rr, R_tt))
        """
        self.q = np.array(q_0, dtype=float)
        self.P = np.array(P_0, dtype=float)
        self.Q = np.array(Q, dtype=float)
        self.R = np.array(R, dtype=float)

    def predict(self, dX: float, dT: float):
        """
        Fase di Predizione: aggiorna posa e covarianza usando l'odometria delle ruote.
        """
        theta = self.q[2]
        
        # Step 1: Propagazione della posa col modello cinematico f
        self.q[0] += dX * np.cos(theta)
        self.q[1] += dX * np.sin(theta)
        self.q[2] = wrap_angle(self.q[2] + dT)
        
        # Step 2: Calcolo delle matrici Jacobiane F (3x3) e W (3x2)
        F = np.array([
            [1.0, 0.0, -dX * np.sin(theta)],
            [0.0, 1.0,  dX * np.cos(theta)],
            [0.0, 0.0,  1.0]
        ])
        
        W = np.array([
            [np.cos(theta), 0.0],
            [np.sin(theta), 0.0],
            [0.0,           1.0]
        ])
        
        # Step 3: Aggiornamento della covarianza predetta
        self.P = F @ self.P @ F.T + W @ self.Q @ W.T

    def update(self, z: np.ndarray, tag_xy: np.ndarray):
        """
        Fase di Aggiornamento: corregge la posa usando la misura di un AprilTag notando (tag_x, tag_y).
        z: [range, bearing]
        tag_xy: [tag_x, tag_y]
        """
        # Step 1: Calcolo della misura predetta [range_pred, bearing_pred]
        dx = tag_xy[0] - self.q[0]
        dy = tag_xy[1] - self.q[1]
        
        rng_pred = np.sqrt(dx**2 + dy**2)
        if rng_pred == 0:
            return
            
        bearing_pred = wrap_angle(np.arctan2(dy, dx) - self.q[2])
        z_pred = np.array([rng_pred, bearing_pred])
        
        # Step 2: Calcolo dell'innovazione (residuo)
        y = z - z_pred
        y[1] = wrap_angle(y[1])
        
        # Step 3: Calcolo della Jacobiana di Misura H (2x3)
        H = np.array([
            [-dx / rng_pred,       -dy / rng_pred,        0.0],
            [ dy / (rng_pred**2),  -dx / (rng_pred**2),  -1.0]
        ])
        
        # Step 4: Calcolo del Guadagno di Kalman K (3x2)
        S = H @ self.P @ H.T + self.R
        K = self.P @ H.T @ np.linalg.inv(S)
        
        # Step 5: Aggiornamento finale di posa e covarianza
        self.q = self.q + K @ y
        self.q[2] = wrap_angle(self.q[2])
        
        I = np.eye(3)
        self.P = (I - K @ H) @ self.P

print("Classe EKF compilata e pronta all'uso!")

---
# Simulazione Completa: Il Duckiebot Naviga tra gli AprilTag

Creiamo uno scenario di simulazione completo per vedere l'EKF all'opera:
* Il robot parte da $(x_0, y_0, \theta_0) = (0.41, 0.61, \pi/2)$ (valori reali del file `default.yaml`).
* Sulla mappa sono posizionati 4 AprilTags.
* Il robot percorre una curva ad arco compiendo 50 passi temporali.
* A ogni passo:
  1. I motori avanzano ($dX = 0.05\text{ m}, dT = 0.04\text{ rad}$).
  2. L'odometria pura (Dead Reckoning) devia progressivamente per via del rumore sulle ruote.
  3. L'EKF esegue la `predict`.
  4. Quando un AprilTag si trova nel cono visivo del robot ($< 2\text{ metri}$), la telecamera misura $[r, \phi]$ con rumore e chiama `update`.
  5. Plottiamo l'**ellisse di covarianza di incertezza 2D** a ogni tappa!

In [ ]:
# Definizione parametri del file default.yaml
q_0 = np.array([0.41, 0.61, 1.57])
P_0 = np.diag([0.1, 0.1, 0.01])
Q = np.diag([0.02, 0.01])       # Rumore processo dX, dT
R = np.diag([0.05, 0.02])       # Rumore sensore r, phi

# Mappa degli AprilTags [x, y]
landmarks = {
    1: np.array([0.68, 1.76]),
    9: np.array([0.61, 1.76]),
    20: np.array([-0.03, 1.83]),
    23: np.array([1.13, 1.79]),
}

# Inizializziamo il filtro EKF
ekf = EKF(q_0, P_0, Q, R)

# Liste per la cronologia della simulazione
steps = 40
gt_traj = [q_0.copy()]
dead_reckoning = [q_0.copy()]
ekf_traj = [q_0.copy()]
cov_history = [ekf.P.copy()]

curr_gt = q_0.copy()
curr_dr = q_0.copy()

# Comandi nominali ad arco
cmd_dX = 0.05
cmd_dT = 0.03

for t in range(steps):
    # 1. Movimento reale (Ground Truth) con slittamento fisico
    true_noise_dX = np.random.normal(0, np.sqrt(Q[0, 0]))
    true_noise_dT = np.random.normal(0, np.sqrt(Q[1, 1]))
    real_dX = cmd_dX + true_noise_dX
    real_dT = cmd_dT + true_noise_dT
    
    curr_gt[0] += real_dX * np.cos(curr_gt[2])
    curr_gt[1] += real_dX * np.sin(curr_gt[2])
    curr_gt[2] = wrap_angle(curr_gt[2] + real_dT)
    gt_traj.append(curr_gt.copy())
    
    # 2. Dead Reckoning (sola integrazione dei comandi senza sensori)
    curr_dr[0] += cmd_dX * np.cos(curr_dr[2])
    curr_dr[1] += cmd_dX * np.sin(curr_dr[2])
    curr_dr[2] = wrap_angle(curr_dr[2] + cmd_dT)
    dead_reckoning.append(curr_dr.copy())
    
    # 3. EKF PREDICT
    ekf.predict(cmd_dX, cmd_dT)
    
    # 4. EKF UPDATE: controlliamo se qualche AprilTag e visibile
    for tag_id, tag_pos in landmarks.items():
        dx = tag_pos[0] - curr_gt[0]
        dy = tag_pos[1] - curr_gt[1]
        true_range = np.sqrt(dx**2 + dy**2)
        true_bearing = wrap_angle(np.arctan2(dy, dx) - curr_gt[2])
        
        # Se il tag e entro 1.5 metri, la telecamera lo legge con rumore
        if true_range < 1.5:
            noisy_range = true_range + np.random.normal(0, np.sqrt(R[0, 0]))
            noisy_bearing = wrap_angle(true_bearing + np.random.normal(0, np.sqrt(R[1, 1])))
            z = np.array([noisy_range, noisy_bearing])
            ekf.update(z, tag_pos)
            
    ekf_traj.append(ekf.q.copy())
    cov_history.append(ekf.P.copy())

gt_traj = np.array(gt_traj)
dead_reckoning = np.array(dead_reckoning)
ekf_traj = np.array(ekf_traj)
print(f"Simulazione completata per {steps} passi temporali!")

In [ ]:
def plot_covariance_ellipse(x, y, P_xy, ax, n_std=2.0, color='g'):
    """Disegna l'ellisse di incertezza (a 2 deviazioni standard) attorno alla posa stimata."""
    vals, vecs = np.linalg.eigh(P_xy[:2, :2])
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    theta = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
    width, height = 2.0 * n_std * np.sqrt(np.maximum(vals, 1e-6))
    ellipse = patches.Ellipse(xy=(x, y), width=width, height=height, angle=theta,
                              edgecolor=color, facecolor=color, alpha=0.15, linewidth=1.5)
    ax.add_patch(ellipse)

fig, ax = plt.subplots(figsize=(10, 10))

# 1. Tracciamento traiettorie
ax.plot(gt_traj[:, 0], gt_traj[:, 1], 'k--', linewidth=2.5, label='Ground Truth Reale')
ax.plot(dead_reckoning[:, 0], dead_reckoning[:, 1], 'b:', linewidth=2, label='Odometria pura (Dead Reckoning)')
ax.plot(ekf_traj[:, 0], ekf_traj[:, 1], 'g-o', markersize=4, linewidth=2, label='Stima EKF Localization')

# 2. AprilTags sulla mappa
for tag_id, tag_pos in landmarks.items():
    ax.scatter(tag_pos[0], tag_pos[1], color='magenta', marker='s', s=150, zorder=6)
    ax.text(tag_pos[0] + 0.03, tag_pos[1] + 0.03, f"Tag #{tag_id}", fontsize=11, fontweight='bold', color='purple')

# 3. Disegno ellissi di covarianza ad intervalli regolari
for i in range(0, len(ekf_traj), 8):
    plot_covariance_ellipse(ekf_traj[i, 0], ekf_traj[i, 1], cov_history[i], ax, n_std=2.0, color='green')

ax.set_title('Localizzazione EKF con AprilTags ed Ellissi di Covarianza (P)', fontsize=14)
ax.set_xlabel('X nel Mondo [metri]', fontsize=12)
ax.set_ylabel('Y nel Mondo [metri]', fontsize=12)
ax.legend(fontsize=11, loc='upper left')
ax.grid(True, linestyle=':', alpha=0.6)
ax.set_aspect('equal')
plt.show()

---
# Consigli Pratici per il Debugging del Nodo ROS (`ekf_localization_node`)

Quando eseguirai il tuo codice nel simulatore **Duckiematrix** o sul vero Duckiebot, se hai commesso un piccolo errore di segno o indice nelle matrici Jacobiane, la stima **divergerà istantaneamente** (il robot volerà a infinito!).

Per fare un debugging efficace:

1. **Isolare il Predict dal Update (`no_update: True`):**
   * Nel file `packages/ekf_localization/config/ekf_localization_node/default.yaml`, imposta temporaneamente:
     ```yaml
     no_update: True
     no_predict: False
     ```
   * In questo modo disattivi le correzioni degli AprilTag. Il Duckiebot seguirà solo la cinematica degli encoder.
   * Verifica che il robot si muova nella direzione giusta. Una volta accertato che il *Predict* funziona, riattiva `no_update: False`.

2. **Isolare il Predict (`no_predict: True`):**
   * Impostando `no_predict: True`, il filtro non userà gli encoder e si aggiornerà solo quando vede un AprilTag.
   * *(Attenzione: se non ci sono cartelli nel campo visivo, la stima rimarrà congelata).*

3. **Verificare la Regola della Mano Destra e gli Angoli:**
   * Ricorda sempre che $\theta$ cresce in senso antiorario.
   * Assicurati di applicare `wrap_angle` sia all'orientamento del robot sia alla coordinata angolare dell'innovazione $y[1]$.

---
# La Grande Sintesi dei 4 Filtri di Localizzazione

Complimenti! Hai completato lo studio dei quattro algoritmi cardine della robotica autonoma:

| Filtro | Tipo di Stato | Tipo di Distribuzione | Punti di Forza | Limiti Principali |
| :--- | :--- | :--- | :--- | :--- |
| **1. Kalman Filter (KF)** | Continuo | Gaussiana unimodale | Ottimo matematicamente, velocissimo | Solo sistemi lineari |
| **2. Particle Filter (PF)** | Continuo | Qualsiasi forma (Monte Carlo) | Gestisce distribuzioni multimodali e robot rapito | Computazionalmente oneroso ($M$ alto) |
| **3. Histogram Filter** | Discreto (Griglia) | Discreta su caselle | Perfetto per compiti circoscritti (Lane Following) | Esplosione esponenziale della memoria in 3D+ |
| **4. Extended Kalman (EKF)**| Continuo | Gaussiana linearizzata (Taylor) | Lo standard industriale globale per navigazione SLAM e GPS | Può divergere se le non linearità sono estreme |

Ora possiedi una visione chiara, profonda e completa di come un robot percepisce l'ambiente, gestisce l'incertezza e naviga con precisione nel mondo reale!